In [4]:
import requests
import pandas as pd

url = "https://dadosabertos.camara.leg.br/api/v2/deputados"
headers = {"Accept": "application/json"}
params = {
    "itens": 100,
    "ordem": "ASC",
    "ordenarPor": "nome"
}

todos_deputados = []

with requests.Session() as session:
    while url:
        response = session.get(url, headers=headers, params=params, timeout=30)
        response.raise_for_status()
        payload = response.json()
        
        todos_deputados.extend(payload.get("dados", []))
        
        # Busca o link da próxima página; se não existir, encerra o loop
        links = payload.get("links", [])
        url = next((link["href"] for link in links if link.get("rel") == "next"), None)
        
        # Os parâmetros já vêm embutidos na URL do link 'next'
        params = None

df_deputados = pd.DataFrame(todos_deputados)
#display(df_deputados.head(10))

In [5]:
import time
import requests
import pandas as pd
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

headers = {"Accept": "application/json"}

# Configura retry automático para erro 429 e instabilidades 5xx
retry_strategy = Retry(
    total=5,
    backoff_factor=2,
    status_forcelist=[429, 500, 502, 503, 504],
    allowed_methods=["GET"]
)

session = requests.Session()
session.mount("https://", HTTPAdapter(max_retries=retry_strategy))

# Verifica se já existe algum registro carregado na memória
ja_existe = "df_deputados_detalhes" in globals() and isinstance(df_deputados_detalhes, pd.DataFrame)
ids_ja_extraidos = set(df_deputados_detalhes["id"].tolist()) if ja_existe else set()
ids_faltantes = [dep_id for dep_id in df_deputados["id"].unique() if dep_id not in ids_ja_extraidos]

print(f"Faltam {len(ids_faltantes)} deputados para extrair...")

novos_detalhes = []

for i, dep_id in enumerate(ids_faltantes, start=1):
    url = f"https://dadosabertos.camara.leg.br/api/v2/deputados/{dep_id}"
    try:
        resp = session.get(url, headers=headers, timeout=30)
        resp.raise_for_status()
        dados = resp.json().get("dados")
        if dados:
            novos_detalhes.append(dados)
    except requests.RequestException as e:
        print(f"Falha definitiva no ID {dep_id}: {e}")
    
    time.sleep(0.25)
    
    if i % 50 == 0:
        print(f"Progresso: {i}/{len(ids_faltantes)} processados...")

# Normaliza os novos registros e cria ou consolida o DataFrame
if novos_detalhes:
    df_novos = pd.json_normalize(novos_detalhes, sep="_")
    if "redeSocial" in df_novos.columns:
        df_novos["redeSocial"] = df_novos["redeSocial"].apply(
            lambda x: ", ".join(x) if isinstance(x, list) else None
        )
    
    if ja_existe:
        df_deputados_detalhes = pd.concat([df_deputados_detalhes, df_novos], ignore_index=True)
    else:
        df_deputados_detalhes = df_novos

# print(f"Total consolidado: {len(df_deputados_detalhes)} de {len(df_deputados)} deputados.")
# display(df_deputados_detalhes.head())

Faltam 0 deputados para extrair...


In [ ]:
import sys
from pathlib import Path
from datetime import datetime
import pandas as pd
import psycopg2
from psycopg2.extras import execute_values

# 1. Garante que a raiz do projeto esteja no sys.path e importa as credenciais do main.py
root_path = Path.cwd().resolve().parents[1]
if str(root_path) not in sys.path:
    sys.path.append(str(root_path))

from main import db_host, db_name, db_password, db_port, db_schema, db_user

# 2. Mapeia as 32 colunas para o padrão Bronze
mapeamento_bronze = {
    "id": "n_id_deputado",
    "uri": "s_ds_uri",
    "nomeCivil": "s_ds_nome_civil",
    "cpf": "s_cd_cpf",
    "sexo": "s_cd_sexo",
    "urlWebsite": "s_ds_url_website",
    "redeSocial": "s_ds_rede_social",
    "dataNascimento": "s_dt_nascimento",
    "dataFalecimento": "s_dt_falecimento",
    "ufNascimento": "s_cd_uf_nascimento",
    "municipioNascimento": "s_ds_municipio_nascimento",
    "escolaridade": "s_ds_escolaridade",
    "ultimoStatus_id": "n_id_ultimo_status",
    "ultimoStatus_uri": "s_ds_ultimo_status_uri",
    "ultimoStatus_nome": "s_ds_nome_parlamentar",
    "ultimoStatus_siglaPartido": "s_cd_sigla_partido",
    "ultimoStatus_uriPartido": "s_ds_uri_partido",
    "ultimoStatus_siglaUf": "s_cd_sigla_uf",
    "ultimoStatus_idLegislatura": "n_id_legislatura",
    "ultimoStatus_urlFoto": "s_ds_url_foto",
    "ultimoStatus_email": "s_ds_email",
    "ultimoStatus_data": "s_dh_ultimo_status",
    "ultimoStatus_nomeEleitoral": "s_ds_nome_eleitoral",
    "ultimoStatus_gabinete_nome": "s_ds_gabinete_nome",
    "ultimoStatus_gabinete_predio": "s_ds_gabinete_predio",
    "ultimoStatus_gabinete_sala": "s_ds_gabinete_sala",
    "ultimoStatus_gabinete_andar": "s_ds_gabinete_andar",
    "ultimoStatus_gabinete_telefone": "s_ds_gabinete_telefone",
    "ultimoStatus_gabinete_email": "s_ds_gabinete_email",
    "ultimoStatus_situacao": "s_ds_situacao",
    "ultimoStatus_condicaoEleitoral": "s_ds_condicao_eleitoral",
    "ultimoStatus_descricaoStatus": "s_ds_descricao_status"
}

df_bronze_detalhes = df_deputados_detalhes.rename(columns=mapeamento_bronze).copy()

# Na camada Bronze mantemos os dados brutos da API, apenas convertendo vazios/NaN para None (NULL no Postgres)
df_bronze_detalhes = df_bronze_detalhes.replace("", None)
df_bronze_detalhes = df_bronze_detalhes.astype(object).where(pd.notnull(df_bronze_detalhes), None)
df_bronze_detalhes["d_dh_carga"] = datetime.now()

colunas_ordenadas = list(mapeamento_bronze.values()) + ["d_dh_carga"]
df_bronze_detalhes = df_bronze_detalhes[colunas_ordenadas]

# 3. DDL da tabela bronze_deputados_detalhes (recria do zero)
ddl_bronze = f"""
DROP TABLE IF EXISTS {db_schema}.bronze_deputados_detalhes;

CREATE TABLE {db_schema}.bronze_deputados_detalhes (
    n_id_deputado               INTEGER PRIMARY KEY,
    s_ds_uri                    TEXT,
    s_ds_nome_civil             VARCHAR(200),
    s_cd_cpf                    VARCHAR(14),
    s_cd_sexo                   CHAR(1),
    s_ds_url_website            TEXT,
    s_ds_rede_social            TEXT,
    s_dt_nascimento             VARCHAR(20),
    s_dt_falecimento            VARCHAR(20),
    s_cd_uf_nascimento          CHAR(2),
    s_ds_municipio_nascimento   VARCHAR(150),
    s_ds_escolaridade           VARCHAR(100),
    n_id_ultimo_status          INTEGER,
    s_ds_ultimo_status_uri      TEXT,
    s_ds_nome_parlamentar       VARCHAR(150),
    s_cd_sigla_partido          VARCHAR(20),
    s_ds_uri_partido            TEXT,
    s_cd_sigla_uf               CHAR(2),
    n_id_legislatura            INTEGER,
    s_ds_url_foto               TEXT,
    s_ds_email                  VARCHAR(150),
    s_dh_ultimo_status          VARCHAR(30),
    s_ds_nome_eleitoral         VARCHAR(150),
    s_ds_gabinete_nome          VARCHAR(50),
    s_ds_gabinete_predio        VARCHAR(50),
    s_ds_gabinete_sala          VARCHAR(50),
    s_ds_gabinete_andar         VARCHAR(50),
    s_ds_gabinete_telefone      VARCHAR(50),
    s_ds_gabinete_email         VARCHAR(150),
    s_ds_situacao               VARCHAR(100),
    s_ds_condicao_eleitoral     VARCHAR(100),
    s_ds_descricao_status       TEXT,
    d_dh_carga                  TIMESTAMP DEFAULT CURRENT_TIMESTAMP
);
"""

cols_str = ", ".join(colunas_ordenadas)
insert_bronze = f"""
INSERT INTO {db_schema}.bronze_deputados_detalhes ({cols_str})
VALUES %s;
"""

registros = [tuple(x) for x in df_bronze_detalhes.itertuples(index=False, name=None)]

# 4. Executa no PostgreSQL (Neon)
with psycopg2.connect(
    host=db_host,
    dbname=db_name,
    user=db_user,
    password=db_password,
    port=db_port,
    sslmode="require"
) as conn:
    with conn.cursor() as cur:
        cur.execute(ddl_bronze)
        execute_values(cur, insert_bronze, registros)
    conn.commit()

# print(f"Sucesso! Tabela recriada e {len(registros)} registros gravados em '{db_schema}.bronze_deputados_detalhes'.")
# display(df_bronze_detalhes.head(5))

Sucesso! Tabela recriada e 513 registros gravados em 'data_challenges_fase_1.bronze_deputados_detalhes'.


,n_id_deputado,s_ds_uri,s_ds_nome_civil,s_cd_cpf,s_cd_sexo,s_ds_url_website,s_ds_rede_social,s_dt_nascimento,s_dt_falecimento,s_cd_uf_nascimento,...,s_ds_gabinete_nome,s_ds_gabinete_predio,s_ds_gabinete_sala,s_ds_gabinete_andar,s_ds_gabinete_telefone,s_ds_gabinete_email,s_ds_situacao,s_ds_condicao_eleitoral,s_ds_descricao_status,d_dh_carga
0,204379,https://dadosabertos.camara.leg.br/api/v2/depu...,ACÁCIO DA SILVA FAVACHO NETO,74287028287,M,None,"https://twitter.com/acaciofavacho, https://www...",1983-09-28,None,AP,...,414,4,414,4,3215-5414,dep.acaciofavacho@camara.leg.br,Exercício,Titular,None,2026-11-26 10:49:41.904764
1,220714,https://dadosabertos.camara.leg.br/api/v2/depu...,ADAIL JOSÉ FIGUEIREDO PINHEIRO,77267796249,M,None,https://twitter.com/adailfilhoam?s=21&t=O_eoTX...,1992-02-16,None,AM,...,531,4,531,5,3215-5531,dep.adailfilho@camara.leg.br,Exercício,Titular,None,2026-11-26 10:49:41.904764
2,221328,https://dadosabertos.camara.leg.br/api/v2/depu...,ADILSON BARROSO OLIVEIRA,05585378805,M,None,None,1964-06-14,None,MG,...,603,4,603,6,3215-5603,dep.adilsonbarroso@camara.leg.br,Exercício,Efetivado,None,2026-11-26 10:49:41.904764
3,204560,https://dadosabertos.camara.leg.br/api/v2/depu...,ADOLFO VIANA DE CASTRO NETO,80123848504,M,None,None,1981-02-02,None,BA,...,911,4,911,9,3215-5911,dep.adolfoviana@camara.leg.br,Exercício,Titular,None,2026-11-26 10:49:41.904764
4,204528,https://dadosabertos.camara.leg.br/api/v2/depu...,ADRIANA MIGUEL VENTURA,12519851813,F,None,"https://twitter.com/adriventurasp, https://www...",1969-03-06,None,SP,...,802,4,802,8,3215-5802,dep.adrianaventura@camara.leg.br,Exercício,Titular,None,2026-11-26 10:49:41.904764
